### Transformar Datos "Employees" - String a JSON
1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos
2. Transformar cadena JSON en objeto JSON
3. Escribir los datos "transformados" en el schema "silver"

In [0]:
SELECT * 
FROM zenviro.bronze.v_employees;

#### 1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos

In [0]:
SELECT value,
       regexp_replace(value, '"birth_date": (\\d{4}-\\d{2}-\\d{2})', '"birth_date": "\$1"') AS fixed_value
FROM zenviro.bronze.v_employees;

In [0]:
CREATE OR REPLACE TEMP VIEW tv_employees_fixed
AS
  SELECT value,
        regexp_replace(value, '"birth_date": (\\d{4}-\\d{2}-\\d{2})', '"birth_date": "\$1"') AS fixed_value
  FROM zenviro.bronze.v_employees;

#### 2. Transformar cadena JSON en objeto JSON

In [0]:
SELECT schema_of_json(fixed_value) AS schema,
       fixed_value
FROM tv_employees_fixed
LIMIT 1;

In [0]:
SELECT from_json(fixed_value,
                'STRUCT<birth_date: STRING, department_id: BIGINT, email: STRING, employee_id: BIGINT, gender: STRING, hire_date: STRING, job_position_id: BIGINT, name: ARRAY<STRUCT<first_name: STRING, last_name: STRING>>, phone: STRING, status: BIGINT>') AS json_value,
                fixed_value
FROM tv_employees_fixed

#### 3. Escribir los datos "transformados" en el schema "silver"

In [0]:
-- DROP TABLE IF EXISTS zenviro.silver.employees_json;
CREATE OR REPLACE TABLE zenviro.silver.employees_json
AS
SELECT from_json(fixed_value,
                'STRUCT<birth_date: STRING, department_id: BIGINT, email: STRING, employee_id: BIGINT, gender: STRING, hire_date: STRING, job_position_id: BIGINT, name: ARRAY<STRUCT<first_name: STRING, last_name: STRING>>, phone: STRING, status: BIGINT>') AS json_value,
                fixed_value
FROM tv_employees_fixed

In [0]:
SELECT * FROM zenviro.silver.employees_json;